# RAG Jour 3
## Reponse au question assurer par un LLM

In [47]:
# imports
import os
import glob
import numpy as np
import gradio as gr
import plotly.graph_objects as go
from sklearn.manifold import TSNE
from dotenv import load_dotenv

from langchain_postgres import PGVector
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace, HuggingFaceEmbeddings

In [48]:
# Load environment variables in a file called .env

load_dotenv()
db_name = "vector_db"

In [ ]:
# Utilisation du SDK de OpenAI avec l'API de Groq
groq_api_key = os.getenv('GROQ_API_KEY') 

MODEL = "llama-4-maverick-70b"
groq_model="meta-llama/llama-4-scout-17b-16e-instruct"

In [50]:
# Utilisation du SDK de OpenAI avec l"API de Hugging Face
hf_api_key = os.getenv('HF_TOKEN')

hf_model = 'meta-llama/Meta-Llama-3.1-8B-Instruct'

## Connection a Chroma BD avec HF

In [8]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma(collection_name=db_name, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Connection a PostgrSQL avec Hugging Face All-MiniLM-L6-v2

In [51]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
connection = os.getenv('DATABASE_URL') 
COLLECTION_NAME = "insurance_knowledge_base"

vector_store = PGVector(
    embeddings=embeddings,
    collection_name=COLLECTION_NAME,
    connection=connection,
    use_jsonb=True # Recommandé pour la performance sur les métadonnées
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Etape 2: decoder et LLM

In [64]:
retriever = vector_store.as_retriever()
groq_llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct", 
    temperature=0,
    api_key=groq_api_key
)

In [53]:
llm_engine = HuggingFaceEndpoint(
    repo_id=hf_model,
    task="text-generation",
    huggingfacehub_api_token=hf_api_key, 
    temperature=0.1,
)
hf_llm = ChatHuggingFace(llm=llm_engine)
retriever = vector_store.as_retriever()

## Implemantation de la methode invoke() de LangChain

In [65]:
retriever.invoke("Who is Jordan")

[Document(id='5ca6c38c-81c4-4484-b859-fcaea0d43241', metadata={'source': 'knowledge-base/employees/Jordan Blake.md', 'doc_type': 'employees'}, page_content='## Other HR Notes\n- Jordan has shown an interest in continuing education, actively participating in company-sponsored sales webinars.  \n- Notable for involvement in the Insurellm volunteer program, assisting local charity events related to financial literacy.  \n- Employee wellness advocate, consistently promotes team bonding activities and stress-relief workshops.  \n- Plans to enroll in a course for advanced sales strategies in Q4 2023, aiming to further enhance his skills at Insurellm.'),
 Document(id='5a9a8ef3-1862-4fdf-bd77-a87dbdedeb15', metadata={'source': 'knowledge-base/employees/Jordan Blake.md', 'doc_type': 'employees'}, page_content='# HR Record\n\n# Jordan Blake\n\n## Summary\n- **Date of Birth:** March 15, 1993  \n- **Job Title:** Sales Development Representative (SDR)  \n- **Location:** Austin, Texas  \n\n## Insure

In [66]:
groq_llm.invoke("Who is Jordan ?")

AIMessage(content='There are several notable individuals named Jordan, so it\'s possible that you\'re referring to one of them. Here are a few examples:\n\n1. **Michael Jordan**: A former American professional basketball player and entrepreneur. He is widely regarded as one of the greatest basketball players of all time, and his success on and off the court has made him a global sports icon.\n2. **Jordan (country)**: Jordan, officially known as the Hashemite Kingdom of Jordan, is a country located in the Middle East. It is bordered by Syria to the north, Iraq to the northeast, Israel and Palestine to the west, and Saudi Arabia to the south and east.\n3. **Jordan (name)**: Jordan is also a given name, commonly used for both males and females. It\'s of Hebrew origin, meaning "flowing down" or " descending".\n\nIf you could provide more context or clarify which Jordan you\'re referring to, I\'d be happy to try and provide more information!', additional_kwargs={}, response_metadata={'token

In [67]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowlegeable, frinedly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [68]:
def answer_question(question: str, histoory):
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = groq_llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [69]:
answer_question("Who is Jordan ?", [])

"Jordan Blake is an employee at Insurellm. He is a Sales Development Representative (SDR) based in Austin, Texas, and was born on March 15, 1993. He has been with Insurellm since June 2021 and has had a notable career progression, including a promotion to Junior SDR and recognition as SDR of the Month for three consecutive months. He's also actively involved in company-sponsored sales webinars, the Insurellm volunteer program, and employee wellness initiatives."

## Quels sont les prochaines possibilites a venir

In [ ]:
gr.ChatInterface(answer_question).launch()